In [1]:
import sys
import numpy as np

sys.path.append("../../../src/")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-04 20:13:02.510718: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-04 20:13:08.543549: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "onPremise",
      "params": {
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'],
        "ports": [50151, 50152, 50153]
      }
      
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-04 20:15:22,128 [DEBUG] [Rain] Rain is initialized
2023-07-04 20:15:22,130 [DEBUG] [Provisioner] Creating coordinator
2023-07-04 20:15:22,131 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/coord/
2023-07-04 20:15:22,133 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-04 20:15:22,134 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-04 20:15:22,135 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/divider/
2023-07-04 20:15:22,136 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/divider/
2023-07-04 20:15:22,137 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/divider/


['127.0.0.1', '127.0.0.1', '127.0.0.1']
[50151, 50152, 50153]


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-04 20:15:29,520 [DEBUG] [Rain] Creating workers
2023-07-04 20:15:29,846 [INFO] [Provisioner] provisioner is serving
2023-07-04 20:15:29,849 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 20:15:29,855 [INFO] [Coordinator] coordinator is serving
2023-07-04 20:15:29,859 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 20:15:29,956 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 20:15:29,960 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 20:15:30,005 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 20:15:30,010 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/worker/
2023-07-04 20:15:30,017 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 20:15:30,022 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../..//RainData/worker/
2023-07-04 20:15:30,029 [INFO] [Worker_

157/157 [==============================] - 2s 7ms/step - loss: 0.7013 - accuracy: 0.7790
sending data to divider


2023-07-04 20:15:53,888 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-04 20:15:53,891 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-04 20:15:54,154 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 20:15:54,168 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3
2023-07-04 20:15:54,238 [DEBUG] [DeepLearning] Iteration 1/3 complete for worker 3.
2023-07-04 20:15:54,238 [DEBUG] [DeepLearning] Starting iteration 2/3
2023-07-04 20:15:54,239 [DEBUG] [DividerAmbassador] 127.0.0.1:50153
2023-07-04 20:15:54,240 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration2 to worker3
2023-07-04 20:15:54,241 [DEBUG] [DividerAmbassador] Sending ../../..//RainData/divider/3.pkl to worker3
2023-07-04 20:15:56,553 [DEBUG] [DividerAmbassador] divider received: File downloa

Error in receiving the gradients from the workers: Ran out of input
Error in receiving the gradients from the workers: Ran out of input


2023-07-04 20:16:12,990 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 1
DEBUG:DividerAmbassador:divider received: File downloaded successfully after sending the model to worker 1
2023-07-04 20:16:12,993 [DEBUG] [DividerAmbassador] divider begins executing iteration3 for worker1
DEBUG:DividerAmbassador:divider begins executing iteration3 for worker1
2023-07-04 20:16:13,017 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 2
DEBUG:DividerAmbassador:divider received: File downloaded successfully after sending the model to worker 2
2023-07-04 20:16:13,019 [DEBUG] [DividerAmbassador] divider begins executing iteration3 for worker2
DEBUG:DividerAmbassador:divider begins executing iteration3 for worker2
2023-07-04 20:16:15,850 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Exe

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 3ms/step - loss: 0.1943 - accuracy: 0.9433

Test accuracy: 94.3%


In [ ]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

2023-07-04 16:28:46,565 [DEBUG] [Rain] Creating workers
2023-07-04 16:28:46,576 [INFO] [Provisioner] provisioner is serving
2023-07-04 16:28:46,577 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 16:28:46,579 [INFO] [Coordinator] coordinator is serving
2023-07-04 16:28:46,581 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 16:28:46,590 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 16:28:46,593 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 16:28:46,595 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 16:28:46,596 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData\worker/
2023-07-04 16:28:46,599 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 16:28:46,600 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData\worker/
2023-07-04 16:28:46,603 [INFO] [W

  7/157 [>.............................] - ETA: 1s - loss: 2.1619 - accuracy: 0.2377  

2023-07-04 16:29:29,765 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 2
2023-07-04 16:29:29,768 [DEBUG] [DividerAmbassador] divider begins executing iteration1 for worker2
2023-07-04 16:29:29,778 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 1


157/157 [==============================] - 4s 12ms/step - loss: 0.6954 - accuracy: 0.7820
sending data to coordinator

2023-07-04 16:29:31,608 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-04 16:29:31,616 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData\divider/3_1_trained.pkl from worker3


...

Error in loading the data:  Ran out of input
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-04 16:29:37,317 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData\divider/2_2_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData\divider/2_2_trained.pkl from worker2 successfully


157/157 [==============================] - 3s 9ms/step - loss: 0.3158 - accuracy: 0.9046
sending data to coordinator

2023-07-04 16:29:39,575 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 3
2023-07-04 16:29:39,578 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData\divider/3_2_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData\divider/3_2_trained.pkl from worker3



sending data to coordinator


In [ ]:
# model = rain.train(X_train, y_train, strategy='sync')

In [ ]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-04 16:29:39,600 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 1
2023-07-04 16:29:39,605 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData\divider/1_2_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData\divider/1_2_trained.pkl from worker1
2023-07-04 16:29:40,233 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData\divider/1_2_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData\divider/1_2_trained.pkl from worker1 successfully
2023-07-04 16:29:40,241 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData\divider/3_2_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData\divider/3_2_trained.pkl from worker3 successfully
2023-07-04 16:29:40,277 [DEBUG] [DeepLearning] Iteration 2/3

141/157 [=========================>....] - ETA: 0s - loss: 0.2578 - accuracy: 0.9240sending data to coordinator


2023-07-04 16:29:46,246 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 2
2023-07-04 16:29:46,257 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData\divider/2_3_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData\divider/2_3_trained.pkl from worker2


157/157 [==============================] - 5s 12ms/step - loss: 0.2576 - accuracy: 0.9243


2023-07-04 16:29:46,443 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 3
2023-07-04 16:29:46,453 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData\divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData\divider/3_3_trained.pkl from worker3
2023-07-04 16:29:46,461 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1


sending data to coordinator
sending data to coordinator


DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 1
2023-07-04 16:29:46,471 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData\divider/1_3_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData\divider/1_3_trained.pkl from worker1
2023-07-04 16:29:47,247 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData\divider/2_3_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData\divider/2_3_trained.pkl from worker2 successfully
2023-07-04 16:29:47,334 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData\divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData\divider/3_3_trained.pkl from worker3 successfully
2023-07-04 16:29:47,372 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData\divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..

In [ ]:
# X_test, y_test = get_test_data()
# loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
# print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 2ms/step - loss: 0.1407 - accuracy: 0.9576

Test accuracy: 95.8%
